In [ ]:
%%sql -r result1
CREATE OR REPLACE DATABASE HEALTHCARE_DW;

USE DATABASE HEALTHCARE_DW;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE SCHEMA RAW;

CREATE OR REPLACE SCHEMA DIM;

CREATE OR REPLACE SCHEMA FACT;

CREATE OR REPLACE SCHEMA ANALYTICS;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE WAREHOUSE HEALTHCARE_WH
WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

In [ ]:
%%sql -r dataframe_3
USE WAREHOUSE HEALTHCARE_WH;

In [ ]:
%%sql -r csv_format_result
CREATE SCHEMA IF NOT EXISTS RAW;

CREATE OR REPLACE FILE FORMAT RAW.CSV_FORMAT
TYPE = 'CSV'
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
SKIP_HEADER = 1
NULL_IF = ('NULL', 'null', '');

In [ ]:
%%sql -r dataframe_5
SHOW FILE FORMATS;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE STAGE RAW.HEALTHCARE_STAGE
FILE_FORMAT = RAW.CSV_FORMAT;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE RAW.STG_PATIENTS
(
    PATIENT_ID VARCHAR,
    PATIENT_NAME VARCHAR,
    GENDER VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR
);

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE RAW.STG_DOCTORS
(
    DOCTOR_ID VARCHAR,
    DOCTOR_NAME VARCHAR,
    SPECIALIZATION VARCHAR
);

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE RAW.STG_HOSPITALS
(
    HOSPITAL_ID VARCHAR,
    HOSPITAL_NAME VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR,
    REGION VARCHAR
);

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE RAW.STG_DEPARTMENTS
(
    DEPARTMENT_ID VARCHAR,
    DEPARTMENT_NAME VARCHAR
);

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE TABLE RAW.STG_TREATMENTS
(
    TREATMENT_ID VARCHAR,
    TREATMENT_NAME VARCHAR,
    TREATMENT_CATEGORY VARCHAR
);

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE RAW.STG_ADMISSIONS
(
    ADMISSION_ID VARCHAR,
    PATIENT_ID VARCHAR,
    DOCTOR_ID VARCHAR,
    HOSPITAL_ID VARCHAR,
    DEPARTMENT_ID VARCHAR,
    ADMISSION_DATE DATE,
    DISCHARGE_DATE DATE
);

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE TABLE RAW.STG_BILLING
(
    BILLING_ID VARCHAR,
    PATIENT_ID VARCHAR,
    DOCTOR_ID VARCHAR,
    HOSPITAL_ID VARCHAR,
    DEPARTMENT_ID VARCHAR,
    TREATMENT_ID VARCHAR,
    BILLING_DATE DATE,
    QUANTITY NUMBER,
    TREATMENT_AMOUNT NUMBER(12,2),
    DISCOUNT NUMBER(12,2)
);

In [ ]:
%%sql -r dataframe_13
COPY INTO RAW.STG_PATIENTS
FROM @RAW.HEALTHCARE_STAGE/patients.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_14
COPY INTO RAW.STG_DOCTORS
FROM @RAW.HEALTHCARE_STAGE/doctors.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_15
COPY INTO RAW.STG_HOSPITALS
FROM @RAW.HEALTHCARE_STAGE/hospitals.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_16
COPY INTO RAW.STG_DEPARTMENTS
FROM @RAW.HEALTHCARE_STAGE/departments.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_17
COPY INTO RAW.STG_TREATMENTS
FROM @RAW.HEALTHCARE_STAGE/treatments.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_18
COPY INTO RAW.STG_ADMISSIONS
FROM @RAW.HEALTHCARE_STAGE/admissions.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_19
COPY INTO RAW.STG_BILLING
FROM @RAW.HEALTHCARE_STAGE/billing.csv
FILE_FORMAT = (FORMAT_NAME = RAW.CSV_FORMAT)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r dataframe_20
SELECT COUNT(*) FROM RAW.STG_PATIENTS;

SELECT COUNT(*) FROM RAW.STG_DOCTORS;

SELECT COUNT(*) FROM RAW.STG_HOSPITALS;

SELECT COUNT(*) FROM RAW.STG_DEPARTMENTS;

SELECT COUNT(*) FROM RAW.STG_TREATMENTS;

SELECT COUNT(*) FROM RAW.STG_ADMISSIONS;

SELECT COUNT(*) FROM RAW.STG_BILLING;

In [ ]:
%%sql -r dataframe_23
CREATE OR REPLACE SCHEMA ANALYTICS;

In [ ]:
%%sql -r dataframe_24
CREATE OR REPLACE SCHEMA FACT;

In [ ]:
%%sql -r dataframe_25
CREATE OR REPLACE SCHEMA DIM;


In [ ]:
%%sql -r dataframe_21
CREATE OR REPLACE TABLE DIM.DIM_PATIENT
(
    PATIENT_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    PATIENT_ID VARCHAR,
    PATIENT_NAME VARCHAR,
    GENDER VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR
);

In [ ]:
%%sql -r dataframe_22
INSERT INTO DIM.DIM_PATIENT
(
    PATIENT_ID,
    PATIENT_NAME,
    GENDER,
    CITY,
    STATE
)
SELECT
    PATIENT_ID,
    PATIENT_NAME,
    GENDER,
    CITY,
    STATE
FROM RAW.STG_PATIENTS;

In [ ]:
%%sql -r dataframe_26
SELECT *
FROM DIM.DIM_PATIENT
ORDER BY PATIENT_KEY;

In [ ]:
%%sql -r dataframe_27
CREATE OR REPLACE TABLE DIM.DIM_DOCTOR
(
    DOCTOR_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    DOCTOR_ID VARCHAR,
    DOCTOR_NAME VARCHAR,
    SPECIALIZATION VARCHAR
);

In [ ]:
%%sql -r dataframe_28
INSERT INTO DIM.DIM_DOCTOR
(
    DOCTOR_ID,
    DOCTOR_NAME,
    SPECIALIZATION
)
SELECT
    DOCTOR_ID,
    DOCTOR_NAME,
    SPECIALIZATION
FROM RAW.STG_DOCTORS;

In [ ]:
%%sql -r dataframe_29
CREATE OR REPLACE TABLE DIM.DIM_HOSPITAL
(
    HOSPITAL_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    HOSPITAL_ID VARCHAR,
    HOSPITAL_NAME VARCHAR,
    CITY VARCHAR,
    STATE VARCHAR,
    REGION VARCHAR
);

In [ ]:
%%sql -r dataframe_30
INSERT INTO DIM.DIM_HOSPITAL
(
    HOSPITAL_ID,
    HOSPITAL_NAME,
    CITY,
    STATE,
    REGION
)
SELECT
    HOSPITAL_ID,
    HOSPITAL_NAME,
    CITY,
    STATE,
    REGION
FROM RAW.STG_HOSPITALS;

In [ ]:
%%sql -r dataframe_31
CREATE OR REPLACE TABLE DIM.DIM_DEPARTMENT
(
    DEPARTMENT_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    DEPARTMENT_ID VARCHAR,
    DEPARTMENT_NAME VARCHAR
);

In [ ]:
%%sql -r dataframe_32
INSERT INTO DIM.DIM_DEPARTMENT
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME
)
SELECT
    DEPARTMENT_ID,
    DEPARTMENT_NAME
FROM RAW.STG_DEPARTMENTS;

In [ ]:
%%sql -r dataframe_33
CREATE OR REPLACE TABLE DIM.DIM_TREATMENT
(
    TREATMENT_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    TREATMENT_ID VARCHAR,
    TREATMENT_NAME VARCHAR,
    TREATMENT_CATEGORY VARCHAR
);

In [ ]:
%%sql -r dataframe_35
INSERT INTO DIM.DIM_TREATMENT
(
    TREATMENT_ID,
    TREATMENT_NAME,
    TREATMENT_CATEGORY
)
SELECT
    TREATMENT_ID,
    TREATMENT_NAME,
    TREATMENT_CATEGORY
FROM RAW.STG_TREATMENTS;

In [ ]:
%%sql -r dataframe_36
CREATE OR REPLACE TABLE DIM.DIM_DATE
(
    DATE_KEY NUMBER,
    FULL_DATE DATE,
    DAY NUMBER,
    DAY_NAME VARCHAR,
    WEEK_NO NUMBER,
    MONTH NUMBER,
    MONTH_NAME VARCHAR,
    QUARTER VARCHAR,
    YEAR NUMBER
);

In [ ]:
%%sql -r dataframe_37
INSERT INTO DIM.DIM_DATE
(
    DATE_KEY,
    FULL_DATE,
    DAY,
    DAY_NAME,
    WEEK_NO,
    MONTH,
    MONTH_NAME,
    QUARTER,
    YEAR
)
SELECT
    TO_NUMBER(TO_CHAR(DATE_VALUE, 'YYYYMMDD')) AS DATE_KEY,
    DATE_VALUE AS FULL_DATE,
    DAY(DATE_VALUE) AS DAY,
    DAYNAME(DATE_VALUE) AS DAY_NAME,
    WEEK(DATE_VALUE) AS WEEK_NO,
    MONTH(DATE_VALUE) AS MONTH,
    MONTHNAME(DATE_VALUE) AS MONTH_NAME,
    'Q' || QUARTER(DATE_VALUE) AS QUARTER,
    YEAR(DATE_VALUE) AS YEAR
FROM
(
    SELECT DATEADD(
        DAY,
        SEQ4(),
        '2026-01-01'::DATE
    ) AS DATE_VALUE
    FROM TABLE(GENERATOR(ROWCOUNT => 90))
);

In [ ]:
%%sql -r dataframe_38
SELECT *
FROM DIM.DIM_DATE
ORDER BY FULL_DATE;

In [ ]:
%%sql -r dataframe_39
SELECT COUNT(*)
FROM DIM.DIM_DATE;

In [ ]:
%%sql -r dataframe_40
CREATE OR REPLACE TABLE FACT.FACT_ADMISSION
(
    ADMISSION_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,

    PATIENT_KEY NUMBER,
    DOCTOR_KEY NUMBER,
    HOSPITAL_KEY NUMBER,
    DEPARTMENT_KEY NUMBER,
    DATE_KEY NUMBER,

    ADMISSION_COUNT NUMBER,
    LENGTH_OF_STAY NUMBER
);

In [ ]:
%%sql -r dataframe_41
INSERT INTO FACT.FACT_ADMISSION
(
    PATIENT_KEY,
    DOCTOR_KEY,
    HOSPITAL_KEY,
    DEPARTMENT_KEY,
    DATE_KEY,
    ADMISSION_COUNT,
    LENGTH_OF_STAY
)
SELECT
    p.PATIENT_KEY,
    d.DOCTOR_KEY,
    h.HOSPITAL_KEY,
    dep.DEPARTMENT_KEY,
    dt.DATE_KEY,

    1 AS ADMISSION_COUNT,

    DATEDIFF(
        DAY,
        a.ADMISSION_DATE,
        a.DISCHARGE_DATE
    ) AS LENGTH_OF_STAY

FROM RAW.STG_ADMISSIONS a

JOIN DIM.DIM_PATIENT p
    ON a.PATIENT_ID = p.PATIENT_ID

JOIN DIM.DIM_DOCTOR d
    ON a.DOCTOR_ID = d.DOCTOR_ID

JOIN DIM.DIM_HOSPITAL h
    ON a.HOSPITAL_ID = h.HOSPITAL_ID

JOIN DIM.DIM_DEPARTMENT dep
    ON a.DEPARTMENT_ID = dep.DEPARTMENT_ID

JOIN DIM.DIM_DATE dt
    ON a.ADMISSION_DATE = dt.FULL_DATE;

In [ ]:
%%sql -r dataframe_42
SELECT *
FROM FACT.FACT_ADMISSION
ORDER BY ADMISSION_KEY;

In [ ]:
%%sql -r dataframe_43
SELECT
    COUNT(*) AS TOTAL_ROWS,
    SUM(ADMISSION_COUNT) AS TOTAL_ADMISSIONS,
    SUM(LENGTH_OF_STAY) AS TOTAL_STAY
FROM FACT.FACT_ADMISSION;

In [ ]:
%%sql -r dataframe_44
CREATE OR REPLACE TABLE FACT.FACT_BILLING
(
    BILLING_KEY NUMBER AUTOINCREMENT START 1 INCREMENT 1,

    PATIENT_KEY NUMBER,
    DOCTOR_KEY NUMBER,
    HOSPITAL_KEY NUMBER,
    DEPARTMENT_KEY NUMBER,
    TREATMENT_KEY NUMBER,
    DATE_KEY NUMBER,

    QUANTITY NUMBER,
    TREATMENT_AMOUNT NUMBER(12,2),
    DISCOUNT NUMBER(12,2),
    NET_AMOUNT NUMBER(12,2)
);

In [ ]:
%%sql -r dataframe_45
SELECT
    f.ADMISSION_KEY,
    p.PATIENT_NAME,
    d.DOCTOR_NAME,
    h.HOSPITAL_NAME,
    dep.DEPARTMENT_NAME,
    dt.FULL_DATE,
    f.ADMISSION_COUNT,
    f.LENGTH_OF_STAY
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f

JOIN HEALTHCARE_DW.DIM.DIM_PATIENT p
    ON f.PATIENT_KEY = p.PATIENT_KEY

JOIN HEALTHCARE_DW.DIM.DIM_DOCTOR d
    ON f.DOCTOR_KEY = d.DOCTOR_KEY

JOIN HEALTHCARE_DW.DIM.DIM_HOSPITAL h
    ON f.HOSPITAL_KEY = h.HOSPITAL_KEY

JOIN HEALTHCARE_DW.DIM.DIM_DEPARTMENT dep
    ON f.DEPARTMENT_KEY = dep.DEPARTMENT_KEY

JOIN HEALTHCARE_DW.DIM.DIM_DATE dt
    ON f.DATE_KEY = dt.DATE_KEY

ORDER BY f.ADMISSION_KEY;

In [ ]:
%%sql -r dataframe_46
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(ADMISSION_COUNT) AS TOTAL_ADMISSIONS,
    SUM(LENGTH_OF_STAY) AS TOTAL_LENGTH_OF_STAY
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION;

In [ ]:
%%sql -r dataframe_47
SELECT
    h.HOSPITAL_NAME,
    SUM(f.ADMISSION_COUNT) AS TOTAL_ADMISSIONS
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f
JOIN HEALTHCARE_DW.DIM.DIM_HOSPITAL h
    ON f.HOSPITAL_KEY = h.HOSPITAL_KEY
GROUP BY h.HOSPITAL_NAME
ORDER BY TOTAL_ADMISSIONS DESC;

In [ ]:
%%sql -r dataframe_48
SELECT
    dep.DEPARTMENT_NAME,
    SUM(f.ADMISSION_COUNT) AS TOTAL_ADMISSIONS
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f
JOIN HEALTHCARE_DW.DIM.DIM_DEPARTMENT dep
    ON f.DEPARTMENT_KEY = dep.DEPARTMENT_KEY
GROUP BY dep.DEPARTMENT_NAME
ORDER BY TOTAL_ADMISSIONS DESC;

In [ ]:
%%sql -r dataframe_49
SELECT
    d.DOCTOR_NAME,
    SUM(f.ADMISSION_COUNT) AS TOTAL_ADMISSIONS
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f
JOIN HEALTHCARE_DW.DIM.DIM_DOCTOR d
    ON f.DOCTOR_KEY = d.DOCTOR_KEY
GROUP BY d.DOCTOR_NAME
ORDER BY TOTAL_ADMISSIONS DESC;

In [ ]:
%%sql -r dataframe_50
SELECT
    TO_CHAR(d.FULL_DATE, 'YYYY-MM') AS MONTH,
    SUM(f.ADMISSION_COUNT) AS TOTAL_ADMISSIONS
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f
JOIN HEALTHCARE_DW.DIM.DIM_DATE d
    ON f.DATE_KEY = d.DATE_KEY
GROUP BY TO_CHAR(d.FULL_DATE, 'YYYY-MM')
ORDER BY MONTH;

In [ ]:
%%sql -r dataframe_51
SELECT
    AVG(LENGTH_OF_STAY) AS AVERAGE_LENGTH_OF_STAY
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION;

In [ ]:
%%sql -r dataframe_52
SELECT
    h.HOSPITAL_NAME,
    SUM(f.LENGTH_OF_STAY) AS TOTAL_PATIENT_DAYS,
    AVG(f.LENGTH_OF_STAY) AS AVG_LENGTH_OF_STAY
FROM HEALTHCARE_DW.FACT.FACT_ADMISSION f
JOIN HEALTHCARE_DW.DIM.DIM_HOSPITAL h
    ON f.HOSPITAL_KEY = h.HOSPITAL_KEY
GROUP BY h.HOSPITAL_NAME
ORDER BY TOTAL_PATIENT_DAYS DESC;

In [ ]:
%%sql -r dataframe_53
